In [1]:
import re
import os

In [2]:
from dotenv import load_dotenv

load_dotenv()

token = os.environ.get("HUGGING_TOKEN")

In [3]:
from transformers import pipeline

generator = pipeline("text-generation", model="mistralai/Mistral-7B-Instruct-v0.3", token=token, temperature=0.001)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Device set to use cuda:0


In [4]:
prompt = "5a times 2b equals 10. and a plus b equals 5"
instruction = [
    {
        "role": "system", "content":
            "You are a natural language equation parser. You will receive an equation described in a natural language.\n"
            "1. Output ONLY a comma-separated list of equations. Each equation must:\n"
            "   - Be in single quotes: 'example'\n"
            "   - Consist of two expressions separated by =\n"
            "   - Both of the expressions must consist of named variables, numbers, and operators between them\n"
            "   - The named variables consist of latin characters only, e.g. x, y, var, john, apple etc.\n"
            "   - The numbers should use '.' for decimal points when necessary\n"
            "   - The only operators allowed are: +, -, *, /, ** and there should be spaces on both sides of each operator \n"
            "2. If the input describes an inequality (>, <, >=, <=, !=, or their verbal forms), respond ONLY with: INEQUAL_WARNING.\n"
            "3. If the input does not describe a valid math equation, respond ONLY with: NOTMATH_WARNING.\n"
            "Do not solve the equations. Do not explain anything. Do not output code. Output nothing except what the rules above require."
    },
]

messages = instruction + [
  {"role": "user", "content": "This is the equation described in a natural language:\n<<<\n3 times a plus 4b equals 7\n>>>"},
  {"role": "assistant", "content": "'3 * a + 4 * b = 7'"},
  {"role": "user", "content": "This is the equation described in a natural language:\n<<<\ntwo a minus twentyone equals b. and c squared equals b as well. c=2a\n>>>"},
  {"role": "assistant", "content": "'2 * a - 21 = b', 'c ** 2 = b', c = 2 * a"},
  {"role": "user", "content": "This is the equation described in a natural language:\n<<<\n2x minus five equals zero\n>>>"},
  {"role": "assistant", "content": "'2 * x - 5 = 0'"},
  {"role": "user", "content":
    f"This is the equation described in a natural language:\n<<<\n{prompt}\n>>>"
  },
]

In [5]:
result = generator(messages)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [6]:
from parse_helpers import result_parser

parsed_answer = result_parser(result)

In [7]:
if parsed_answer == 'NOTMATH_WARNING' :
    raise Exception('The prompt was detected not to be a math equation.')

if parsed_answer == 'INEQUAL_WARNING' :
    raise Exception('The prompt was detected to be an inequality.')

In [8]:
from parse_helpers import explicit_multiply_parser

equations = []
for eq in parsed_answer :
    equations.append(explicit_multiply_parser(eq))

print(equations)

[' 5 * a * 2 * b = 10', 'a + b = 5']


In [9]:
from parse_helpers import is_safe_equation

for eq in equations :
    if not is_safe_equation(eq) :
        raise Exception(f"The equation {eq} is not safe.")

In [10]:
symbol_names = set()

for eq in equations :
    matches = re.findall(r"[A-Za-z]+", eq)
    symbol_names.update(matches)

print(f'Symbol names: {symbol_names}')

Symbol names: {'a', 'b'}


In [11]:
from sympy import symbols, Eq, solve, sympify

sympy_symbols = symbols(" ".join(symbol_names))

In [12]:
equation_set = []

for eq in equations :
    left, right = eq.split('=')

    equation_set.append(
        Eq(sympify(left), sympify(right))
    )

In [13]:
solution = solve(equation_set, sympy_symbols, dict=True)

In [14]:
print(solution)

[{a: 5/2 - sqrt(21)/2, b: sqrt(21)/2 + 5/2}, {a: sqrt(21)/2 + 5/2, b: 5/2 - sqrt(21)/2}]
